# AncestryClassifier — Colab Training

Run preprocessing locally with Snakemake first:
```bash
snakemake --cores 4 prepare_training_data simulate_admixed
```
Upload `data/dataset.h5` and `data/admixed_test.h5` to Google Drive, then set `DRIVE_DIR` below.

**After any runtime restart: re-run Cell 1 (config) before running any other cell.**

In [1]:
# ── Cell 1: config — re-run this first after every runtime restart ─────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR     = '/content/drive/MyDrive/gene461'          # <-- change if needed
REPO          = '/content/gene_461_final_project'
WINDOW_SIZE   = 1000

DATA_H5       = f'{DRIVE_DIR}/dataset.h5'
ADMIXED_H5    = f'{DRIVE_DIR}/admixed_test.h5'
CKPT_OUT      = f'{DRIVE_DIR}/checkpoints/best_model.pt'
CONFUSION_OUT = f'{DRIVE_DIR}/confusion_matrix.png'
KARYOGRAM_OUT = f'{DRIVE_DIR}/lai_karyogram.png'

Mounted at /content/drive


In [2]:
# ── Cell 2: one-time setup (clone repo + install deps) ────────────────────
import subprocess, os

if not os.path.exists(REPO):
    subprocess.run(
        ['git', 'clone', 'https://github.com/aszatrowski/gene_461_final_project', REPO],
        check=True
    )
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

%pip install -q torch h5py numpy pandas scikit-learn matplotlib wandb

In [3]:
# ── Cell 3: verify GPU ─────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CUDA available: True
GPU: Tesla T4


In [ ]:
# ── Cell 4: baseline train (needs Cell 1) ─────────────────────────────────
import os
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)

!python {REPO}/scripts/train.py \
    --data        {DATA_H5}    \
    --output      {CKPT_OUT}   \
    --window-size {WINDOW_SIZE} \
    --epochs      25           \
    --batch-size  512          \
    --num-workers 2

Device: cuda
Loading training data ...


^C


In [4]:
# ── Cell 5: W&B login (one-time per session) ───────────────────────────────
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aszatrowski (aszatrowski-university-of-chicago) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
# ── Cell 6: sweep config ───────────────────────────────────────────────────
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {
        'window_size': {'values': [2000, 2500, 5000]},
        'lr':          {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 1e-2},
        'conv_arch':   {'values': [
            # undilated — ERF ~23–55 SNPs
            '32,64_7,5',
            '64,128_7,5',
            '32,64,128_7,5,3',
            # dilated — exponential dilation expands ERF to cover the full window
            '32,64_7,7_1,4',           # ERF ~103 SNPs
            '64,128_7,7_1,4',
            '32,64,128_7,7,7_1,4,16',  # ERF ~1600 SNPs
            '64,128,256_7,7,7_1,4,16',
        ]},
        'global_pool': {'values': [True, False]},
        'dropout':     {'distribution': 'uniform', 'min': 0.1, 'max': 0.5},
    },
}

sweep_id = wandb.sweep(sweep_config, project='ancestry_cnn')
print('Sweep ID:', sweep_id)

Create sweep with ID: gc38pici
Sweep URL: https://wandb.ai/aszatrowski-university-of-chicago/ancestry_cnn/sweeps/gc38pici
Sweep ID: gc38pici


In [ ]:
# ── Cell 7: run sweep agent (needs Cells 1, 3, 5, 6) ─────────────────────
import sys
sys.path.insert(0, f'{REPO}/scripts')
# Evict both modules so Python re-reads the updated files from disk
sys.modules.pop('train', None)
sys.modules.pop('model', None)
from train import run_training, parse_conv_arch

SWEEP_EPOCHS = 10   # short runs during search; best config retrained with 25 epochs below
SWEEP_COUNT  = 20   # total trials (~3-4 h on T4)

def sweep_fn():
    with wandb.init() as run:
        w = dict(run.config)
        channels, kernels, dilations = parse_conv_arch(w['conv_arch'])
        cfg = {
            'window_size':    w['window_size'],
            'lr':             w['lr'],
            'conv_channels':  channels,
            'kernel_sizes':   kernels,
            'dilation_rates': dilations,
            'global_pool':    w['global_pool'],
            'dropout':        w['dropout'],
            'epochs':         SWEEP_EPOCHS,
            'batch_size':     512,
            'num_workers':    2,
            'use_wandb':      True,
        }
        run_training(cfg, DATA_H5, None, device)  # output=None skips checkpoint saving

wandb.agent(sweep_id, sweep_fn, count=SWEEP_COUNT)

wandb: Agent Starting Run: 9fll0ulv with config:
wandb: 	conv_arch: 32,64_7,5
wandb: 	dropout: 0.22078921259929216
wandb: 	global_pool: False
wandb: 	lr: 0.00449887722454982
wandb: 	window_size: 2500
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  train windows: 119,136  |  val windows: 25,568
  params: 2,568,197
Epoch   1/10  loss=2.3934  train_acc=0.2589  val_acc=0.3638
Epoch   2/10  loss=1.5292  train_acc=0.2909  val_acc=0.3457
Epoch   3/10  loss=1.4548  train_acc=0.3125  val_acc=0.3971
Epoch   4/10  loss=1.3913  train_acc=0.3357  val_acc=0.4072
Epoch   5/10  loss=1.3394  train_acc=0.3644  val_acc=0.4894
Epoch   6/10  loss=1.2832  train_acc=0.4041  val_acc=0.5503
Epoch   7/10  loss=1.2347  train_acc=0.4421  val_acc=0.5599
Epoch   8/10  loss=1.2075  train_acc=0.4623  val_acc=0.5897
Epoch   9/10  loss=1.1907  train_acc=0.4739  val_acc=0.5899
Epoch  10/10  loss=1.1729  train_acc=0.4822  val_acc=0.6115

Best val accuracy: 0.6115


epoch,▁▂▃▃▄▅▆▆▇█
loss,█▃▃▂▂▂▁▁▁▁
train_acc,▁▂▃▃▄▆▇▇██
val_acc,▁▁▂▃▅▆▇▇▇█
best_val_acc,0.61147
epoch,10
loss,1.17287
train_acc,0.48225
val_acc,0.61147


wandb: Agent Starting Run: p6yfhih3 with config:
wandb: 	conv_arch: 32,64_7,7_1,4
wandb: 	dropout: 0.3541272596993279
wandb: 	global_pool: False
wandb: 	lr: 0.0027147753357238875
wandb: 	window_size: 2500
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  train windows: 119,136  |  val windows: 25,568
  params: 2,572,293
Epoch   1/10  loss=1.9386  train_acc=0.2550  val_acc=0.3487
Epoch   2/10  loss=1.4803  train_acc=0.2897  val_acc=0.4136
Epoch   3/10  loss=1.4134  train_acc=0.3231  val_acc=0.4301
Epoch   4/10  loss=1.3481  train_acc=0.3631  val_acc=0.5275
Epoch   5/10  loss=1.2838  train_acc=0.4066  val_acc=0.5753
Epoch   6/10  loss=1.2331  train_acc=0.4291  val_acc=0.6126
Epoch   7/10  loss=1.1976  train_acc=0.4453  val_acc=0.6297
Epoch   8/10  loss=1.1736  train_acc=0.4525  val_acc=0.6364
Epoch   9/10  loss=1.1555  train_acc=0.4584  val_acc=0.6437
Epoch  10/10  loss=1.1490  train_acc=0.4611  val_acc=0.6493

Best val accuracy: 0.6493


...

  train windows: 59,568  |  val windows: 12,784
  params: 5,401,861
Epoch   1/10  loss=1.5357  train_acc=0.3177  val_acc=0.4199
Epoch   2/10  loss=1.2977  train_acc=0.4371  val_acc=0.5153
Epoch   3/10  loss=1.1297  train_acc=0.5206  val_acc=0.6909
Epoch   4/10  loss=1.0078  train_acc=0.5749  val_acc=0.7229
Epoch   5/10  loss=0.9327  train_acc=0.6057  val_acc=0.7179
Epoch   6/10  loss=0.8876  train_acc=0.6277  val_acc=0.7365
Epoch   7/10  loss=0.8520  train_acc=0.6456  val_acc=0.7307
Epoch   8/10  loss=0.8312  train_acc=0.6565  val_acc=0.7600
Epoch   9/10  loss=0.8234  train_acc=0.6624  val_acc=0.7656
Epoch  10/10  loss=0.8195  train_acc=0.6611  val_acc=0.7628

Best val accuracy: 0.7656


epoch,▁▂▃▃▄▅▆▆▇█
loss,█▆▄▃▂▂▁▁▁▁
train_acc,▁▃▅▆▇▇████
val_acc,▁▃▆▇▇▇▇███
best_val_acc,0.76564
epoch,10
loss,0.81948
train_acc,0.66111
val_acc,0.76283


wandb: Agent Starting Run: rlq57zq4 with config:
wandb: 	conv_arch: 32,64,128_7,5,3
wandb: 	dropout: 0.4704160780304537
wandb: 	global_pool: True
wandb: 	lr: 0.00030232801348732425
wandb: 	window_size: 2000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  train windows: 148,920  |  val windows: 31,960
  params: 70,021
Epoch   1/10  loss=1.4720  train_acc=0.3203  val_acc=0.4072
Epoch   2/10  loss=1.3491  train_acc=0.3732  val_acc=0.4026


: 

In [7]:
# ── Cell 8: retrain best config for full 25 epochs ────────────────────────
api   = wandb.Api()
sweep = api.sweep(f'aszatrowski-university-of-chicago/ancestry_cnn/{sweep_id}')
best  = max(sweep.runs, key=lambda r: r.summary.get('val_acc', 0))
print('Best run:', best.name)
print('Config:  ', dict(best.config))
print(f'val_acc:  {best.summary["val_acc"]:.4f}')

bc = dict(best.config)
channels, kernels, dilations = parse_conv_arch(bc['conv_arch'])
best_cfg = {
    'window_size':    bc['window_size'],
    'lr':             bc['lr'],
    'conv_channels':  channels,
    'kernel_sizes':   kernels,
    'dilation_rates': dilations,
    'global_pool':    bc['global_pool'],
    'dropout':        bc['dropout'],
    'epochs':         25,
    'batch_size':     512,
    'num_workers':    2,
    'use_wandb':      True,
}
run_training(best_cfg, DATA_H5, CKPT_OUT, device)

Best run: lively-sweep-17
Config:   {'lr': 0.00016165813143785072, 'dropout': 0.32259891952928066, 'conv_arch': '32,64_7,5', 'global_pool': False, 'window_size': 2000}
val_acc:  0.6987
  train windows: 148,920  |  val windows: 31,960
  params: 2,060,293
Epoch   1/25  loss=1.4743  train_acc=0.3381  val_acc=0.4895


Error: You must call wandb.init() before wandb.log()

In [ ]:
# ── Cell 9: evaluate (needs Cell 1 only — safe after a restart) ───────────
!python {REPO}/scripts/evaluate.py \
    --data        {DATA_H5}       \
    --admixed     {ADMIXED_H5}    \
    --checkpoint  {CKPT_OUT}      \
    --confusion   {CONFUSION_OUT} \
    --karyogram   {KARYOGRAM_OUT} \
    --window-size {WINDOW_SIZE}

In [ ]:
# ── Cell 10: display results (needs Cell 1 only) ──────────────────────────
from IPython.display import Image, display
display(Image(CONFUSION_OUT))
display(Image(KARYOGRAM_OUT))